In [1]:
from api import maniskill
from api.video import show
import torch

In [2]:
# StackCube-v1, open-loop (uniform random actions, no trained policy)
TASK = "StackCube-v1"
env = maniskill.make_env(TASK, num_envs=1, render=True)

obs, _ = env.reset(seed=0)
frames = [env.render()[0].cpu().numpy()]
for _ in range(env._max_episode_steps):
    action = torch.from_numpy(env.action_space.sample())
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(env.render()[0].cpu().numpy())
env.close()

show(frames, fps=env.unwrapped.control_freq)

/home/somatic/faraz/control-proj/control/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# PegInsertionSide-v1, open-loop (uniform random actions, no trained policy)
TASK = "PegInsertionSide-v1"
env = maniskill.make_env(TASK, num_envs=1, render=True)

obs, _ = env.reset(seed=0)
frames = [env.render()[0].cpu().numpy()]
for _ in range(env._max_episode_steps):
    action = torch.from_numpy(env.action_space.sample())
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(env.render()[0].cpu().numpy())
env.close()

show(frames, fps=env.unwrapped.control_freq)

In [4]:
# Open-loop replay of a recorded demonstration trajectory, rather than random actions.
TASK = "StackCube-v1"
demo = maniskill.load_demo(TASK)
print(f"{len(demo.actions)} actions, control_mode={demo.control_mode}, recorded success={demo.success}")

107 actions, control_mode=pd_joint_pos, recorded success=True


In [5]:
# Replaying the demo's actions with its exact recorded seed reproduces its exact recorded outcome: the
# simulator is deterministic given (seed, control_mode, backend), and StackCube-v1 randomizes the cube/goal
# poses per-episode from that seed, so matching it reproduces the same poses the actions were planned for.
env = maniskill.make_env(
    TASK, num_envs=1, render=True,
    control_mode=demo.control_mode,
    max_episode_steps=len(demo.actions) + 1,  # demo ran longer than the task's registered 50-step limit
)

obs, _ = env.reset(**demo.reset_kwargs)
frames = [env.render()[0].cpu().numpy()]
for a in demo.actions:
    obs, reward, terminated, truncated, info = env.step(torch.from_numpy(a).unsqueeze(0))
    frames.append(env.render()[0].cpu().numpy())
env.close()

print("replay success:", info["success"].item())
show(frames, fps=env.unwrapped.control_freq)

replay success: True


In [6]:
# PegInsertionSide-v1: open-loop replay of its official motion-planning demo, matched seed (expect success,
# same reasoning as the StackCube-v1 replay above).
PEG_TASK = "PegInsertionSide-v1"
peg_demo = maniskill.load_demo(PEG_TASK)

env = maniskill.make_env(
    PEG_TASK, num_envs=1, render=True,
    control_mode=peg_demo.control_mode,
    max_episode_steps=len(peg_demo.actions) + 1,
)

obs, _ = env.reset(**peg_demo.reset_kwargs)
frames = [env.render()[0].cpu().numpy()]
for a in peg_demo.actions:
    obs, reward, terminated, truncated, info = env.step(torch.from_numpy(a).unsqueeze(0))
    frames.append(env.render()[0].cpu().numpy())
env.close()

print("replay success:", info["success"].item())
show(frames, fps=env.unwrapped.control_freq)

replay success: True


In [7]:
# StackThree-v1 (custom task, api/maniskill_tasks.py): stack cubeB on cubeC, then cubeA on cubeB.
# Sparse reward only: +5 once cubeB is stacked on cubeC, +10 total once the full tower is complete.
# Open-loop random actions here too, so don't expect it to actually build the tower.
TASK = "StackThree-v1"
env = maniskill.make_env(TASK, num_envs=1, render=True)

obs, _ = env.reset(seed=0)
frames = [env.render()[0].cpu().numpy()]
for _ in range(env._max_episode_steps):
    action = torch.from_numpy(env.action_space.sample())
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(env.render()[0].cpu().numpy())
env.close()

print("final reward:", reward.item(), "success:", info["success"].item())
show(frames, fps=env.unwrapped.control_freq)

final reward: 0.0 success: False
